# High-dimensional Bayesian optimization with BAxUS

BAxUS (Bayesian Optimization with Adaptively Expanding Subspaces,
[Papenmeier et al., NeurIPS 2022](https://arxiv.org/abs/2304.11468)) optimizes
high-dimensional problems by working in a low-dimensional random embedding of
the input space. A trust region controls local search in the embedded space;
when it collapses without progress, the embedding *expands*, adding dimensions
until (in the limit) it recovers the full input space. This makes BAxUS
effective when many input dimensions are irrelevant.

This example optimizes a 100-dimensional problem in which only 3 dimensions
actually affect the objective.

**Scope.** Because the model lives in the embedded subspace rather than in
`VOCS` space, this generator is more restricted than the other Bayesian
generators. It handles a single objective over continuous variables and rejects
constraints, observables, discrete and contextual variables, `fixed_features`,
`max_travel_distances`, `custom_objective` and `n_interpolate_points`. It
generates one candidate at a time, and `visualize_model` is not available.

**Differences from the paper.** The acquisition function here is the analytic
LogEI inherited from `ExpectedImprovementGenerator`, evaluated over the trust
region, whereas the paper uses Thompson sampling over a masked discrete
candidate set. The initial subspace dimensionality is the user-set
`target_dim_init` rather than being derived from the input dimensionality.


In [ ]:
import numpy as np

from xopt import Xopt, Evaluator
from xopt.vocs import VOCS
from xopt.generators.bayesian import BAxUSGenerator

N_DIM = 100
N_EFFECTIVE = 3
N_STEPS = 60


def sphere_embedded(inputs: dict) -> dict:
    """Sphere on the first 3 coordinates; the other 97 are irrelevant."""
    x = np.array([inputs[f"x{i}"] for i in range(N_EFFECTIVE)])
    return {"f": float(np.sum(x**2))}


vocs = VOCS(
    variables={f"x{i}": [-1.0, 1.0] for i in range(N_DIM)},
    objectives={"f": "MINIMIZE"},
)
vocs

In [ ]:
generator = BAxUSGenerator(
    vocs=vocs,
    target_dim_init=2,
    seed=0,
    # total planned evaluations: the expansion schedule is paced against this,
    # so it should match the number of steps actually run below
    eval_budget=N_STEPS,
)
evaluator = Evaluator(function=sphere_embedded)
X = Xopt(evaluator=evaluator, generator=generator)
X

## Running the optimization

The generator draws its own Sobol seed points in the embedded space (no
`random_evaluate` needed), then switches to trust-region LogEI. The embedding
expands automatically when the trust region collapses.

In [ ]:
target_dims = []
for _ in range(N_STEPS):
    X.step()
    target_dims.append(X.generator.embedding.target_dim)

print(f"best f: {X.data['f'].min():.3e}")
print(f"embedding grew: {sorted(set(target_dims))}")
X.data[["f"]].tail()

In [ ]:
from matplotlib import pyplot as plt

fig, (ax0, ax1) = plt.subplots(2, 1, sharex=True, figsize=(6, 6))
best = X.data["f"].cummin()
ax0.plot(best.values)
ax0.set_ylabel("best f")
ax0.set_yscale("log")

ax1.step(range(len(target_dims)), target_dims, where="post")
ax1.set_ylabel("embedding target_dim")
ax1.set_xlabel("evaluation")

# mark each embedding expansion on both panels
for i in np.flatnonzero(np.diff(target_dims)) + 1:
    for ax in (ax0, ax1):
        ax.axvline(i, color="grey", ls="--", lw=0.8)
ax0.set_title("BAxUS: convergence and subspace growth")
plt.tight_layout()

## Serialization and resuming

`BAxUSGenerator` owns its own state (the embedding, the trust region, and how
much history has already been folded into it), so it is a `StateOwner`: saving
and reloading through `Xopt` reattaches the data without replaying that history.
Round-tripping a checkpoint therefore leaves the run exactly where it was.


In [ ]:
checkpoint = X.yaml()
resumed = Xopt.from_yaml(checkpoint)

print(
    "target_dim :",
    X.generator.embedding.target_dim,
    "->",
    resumed.generator.embedding.target_dim,
)
print(
    "tr length  :",
    X.generator.trust_region.length,
    "->",
    resumed.generator.trust_region.length,
)
print(
    "rows folded:",
    X.generator.tr_observed_rows,
    "->",
    resumed.generator.tr_observed_rows,
)

resumed.step()  # continues the run
len(resumed.data)

Dumping the generator on its own works too, but a trained generator's dump
carries the live botorch `model` and a `computation_time` frame, neither of
which is YAML-safe -- drop both before serializing by hand.


In [ ]:
import yaml

dump = X.generator.model_dump()
dump.pop("model", None)
dump.pop("computation_time", None)
yaml.safe_dump(dump)[:200]